In [1]:
from functools import partial
from notebooks._utils import report_series_ensemble_accuracy_by_nparas
from notebooks._utils import calculate_parallel_ensemble_accuracy
from notebooks._utils import calculate_baseline_accuracy, get_layers

ds_name = "myriadlama"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

In [4]:
num_fewshots = 0
for model_name in ["llama3.2_1b", "llama3.2_1b_it", "llama3.2_3b", "llama3.2_3b_it", "llama3.1_8b", "llama3.1_8b_it"]:
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    report_accuracy = partial(
        report_series_ensemble_accuracy_by_nparas, 
        dump_file_prefix=dump_file_prefix,
        single_para_qapair=True,
        explicit_prompts=False,
        repeat_paras=False, 
        num_fewshots=num_fewshots, use_generated=True)
    
    print("---- Calculating baseline ----")
    calculate_baseline_accuracy(dataset_root, ds_name, model_name, num_fewshots, is_multichoice=False, by_probs=False)

    report_acc_base_setting = partial(
        calculate_parallel_ensemble_accuracy, 
        dump_file_prefix=dump_file_prefix, repeat_paras=False,
        num_paraphrases=5, num_fewshots=num_fewshots, num_samples=5,
        use_generation=True, by_probs=False, is_multichoice=False)

    print("---- Logits-based Ensemble (Average) ----")
    report_acc_base_setting(logits_ensemble_method="avg")
    
    print("---- Logits-based Ensemble (Maximum) ----")
    report_acc_base_setting(logits_ensemble_method="max")

    print("---- Logits-based Ensemble + Averaged Layer Output ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="layer_output_avg", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))

    print("---- Logits-based Average + Averaged FFN Activation ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_avg", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))

    print("---- Logits-based Average + Maximum FFN Activation ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="avg", ensemble_method="ffn_activation_max", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))

    print("---- Logits-based Maximum + Averaged FFN Activation ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="max", ensemble_method="ffn_activation_avg", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))

    print("---- Logits-based Maximum + Maximum FFN Activation ----")
    multip_layavg_report = partial(report_acc_base_setting, logits_ensemble_method="max", ensemble_method="ffn_activation_max", multilayer=True)
    multip_layavg_report(token_mode="last", ensemble_alpha=1, ensemble_layer=get_layers(model_name))



=================== Model: llama3.2_1b ===================
---- Calculating baseline ----
Acc: 0.1557 ==> 🏷️ baseline of average accuracy per-paraphrase
---- Logits-based Ensemble (Average) ----
Acc: 0.1472 ==> 🏷️ 5paras 0shots  None layerNone  alpha1.0 token-all
---- Logits-based Ensemble (Maximum) ----
Acc: 0.1883 ==> 🏷️ 5paras 0shots  None layerNone  alpha1.0 token-all
---- Logits-based Ensemble + Averaged Layer Output ----
Acc: 0.1900 ==> 🏷️ 5paras 0shots  layer_output_avg layer12 Multilayer alpha1 token-last
---- Logits-based Average + Averaged FFN Activation ----
Acc: 0.1697 ==> 🏷️ 5paras 0shots  ffn_activation_avg layer12 Multilayer alpha1 token-last
---- Logits-based Average + Maximum FFN Activation ----
Acc: 0.1171 ==> 🏷️ 5paras 0shots  ffn_activation_max layer12 Multilayer alpha1 token-last
---- Logits-based Maximum + Averaged FFN Activation ----
Acc: 0.1113 ==> 🏷️ 5paras 0shots  ffn_activation_avg layer12 Multilayer alpha1 token-last
---- Logits-based Maximum + Maximum FFN 